# WBCBench 2026 — Aşama 14A
## Error Analysis + Pre-specified XAI Case Selection

Final-test tahminlerinden PC/BNE/MMY/PMY dahil sınıf hata örüntülerini çıkarır. XAI için vakaları **heatmap sonuçlarını görmeden**, deterministik kategorilerle seçer: tüm modeller doğru, tüm modeller yanlış, yalnızca tek model doğru ve zor sınıf hataları. Böylece XAI örnek seçimi cherry-picking riskini azaltır.

Model inference/eğitim yapmaz.


In [1]:
from pathlib import Path
import csv,json,numpy as np,matplotlib.pyplot as plt
from collections import Counter,defaultdict
from sklearn.metrics import confusion_matrix
R=Path.cwd();I=R/"outputs/final_test_v2";O=R/"outputs/asama14a_error_analysis";O.mkdir(parents=True,exist_ok=True)
T=R/"wbc_project/data/processed/phase2_test"
C=sorted(p.name for p in T.iterdir() if p.is_dir()); M=["convnext_tiny","resnet50","retfound_fullft"]; S=[42,1,7]
EXT={".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}
paths=sorted(p for p in T.rglob("*") if p.is_file() and p.suffix.lower() in EXT)
assert len(paths)==32934
D={}
for m in M:
 for s in S:
  d=I/m/f"seed{s}";D[m,s]=(np.load(d/"true.npy"),np.load(d/"pred.npy"),np.load(d/"probabilities.npy"))
ref=D[M[0],S[0]][0]
print("N=",len(ref),"classes=",C)

# Majority vote per model across seeds; ties resolved by mean probability.
MV={}
for m in M:
 preds=np.stack([D[m,s][1] for s in S]); probs=np.mean(np.stack([D[m,s][2] for s in S]),axis=0)
 out=[]
 for i in range(len(ref)):
  cnt=np.bincount(preds[:,i],minlength=13); mx=cnt.max();cand=np.flatnonzero(cnt==mx)
  out.append(int(cand[0] if len(cand)==1 else cand[np.argmax(probs[i,cand])]))
 MV[m]=np.array(out)

# Support + top confusions for each model.
rows=[]
for m in M:
 cm=confusion_matrix(ref,MV[m],labels=np.arange(13))
 for i,c in enumerate(C):
  order=np.argsort(cm[i])[::-1]
  wrong=[j for j in order if j!=i and cm[i,j]>0][:3]
  rows.append({"model":m,"true_class":c,"support":int((ref==i).sum()),
               "correct":int(cm[i,i]),"recall":float(cm[i,i]/max(1,cm[i].sum())),
               "top_wrong":"; ".join(f"{C[j]}:{cm[i,j]}" for j in wrong)})
with open(O/"class_error_summary.csv","w",newline="",encoding="utf-8") as f:
 w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)
print("\n=== PC / BNE / MMY / PMY ERROR SUMMARY ===")
for r in rows:
 if r["true_class"] in {"PC","BNE","MMY","PMY"}:print(r)

# Systematic XAI case categories. No cherry-picking by heatmap appearance.
all_correct=np.ones(len(ref),bool); all_wrong=np.ones(len(ref),bool)
for m in M:
 all_correct &= MV[m]==ref; all_wrong &= MV[m]!=ref
cases=[]
def add(category,idx,maxn=12):
 for i in idx[:maxn]:
  cases.append({"category":category,"index":int(i),"path":str(paths[i]),"true_class":C[ref[i]],
   **{f"{m}_pred":C[MV[m][i]] for m in M}})
# deterministic ordering by dataset index
add("all_models_correct",np.flatnonzero(all_correct))
add("all_models_wrong",np.flatnonzero(all_wrong))
for m in M:
 only=(MV[m]==ref)
 for other in M:
  if other!=m: only &= MV[other]!=ref
 add(f"only_{m}_correct",np.flatnonzero(only))
for cls in ["PC","BNE","MMY","PMY"]:
 if cls in C:
  ci=C.index(cls); add(f"{cls}_errors",np.flatnonzero((ref==ci)&all_wrong))
with open(O/"xai_case_manifest.csv","w",newline="",encoding="utf-8") as f:
 w=csv.DictWriter(f,fieldnames=cases[0]);w.writeheader();w.writerows(cases)

print("\n=== XAI CASE COUNTS (saved subset max 12/category) ===")
for k,v in Counter(x["category"] for x in cases).items():print(k,v)
print("\nManifest:",(O/"xai_case_manifest.csv").resolve())

# Plot hardest classes by mean majority-vote recall across models
rec=defaultdict(list)
for r in rows:rec[r["true_class"]].append(r["recall"])
means={c:np.mean(v) for c,v in rec.items()}
order=sorted(means,key=means.get)
fig,ax=plt.subplots(figsize=(9,6));ax.barh(order,[means[c] for c in order]);ax.set_xlabel("Mean recall across 3 models");ax.set_title("Class-wise difficulty on final test");fig.tight_layout();fig.savefig(O/"class_difficulty_recall.png",dpi=300,bbox_inches="tight");plt.close(fig)
print("✓ Aşama 14A tamamlandı.")


N= 32934 classes= ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']

=== PC / BNE / MMY / PMY ERROR SUMMARY ===
{'model': 'convnext_tiny', 'true_class': 'BNE', 'support': 252, 'correct': 105, 'recall': 0.4166666666666667, 'top_wrong': 'SNE:120; MO:18; MMY:5'}
{'model': 'convnext_tiny', 'true_class': 'MMY', 'support': 340, 'correct': 182, 'recall': 0.5352941176470588, 'top_wrong': 'BNE:62; SNE:40; MO:23'}
{'model': 'convnext_tiny', 'true_class': 'PC', 'support': 24, 'correct': 0, 'recall': 0.0, 'top_wrong': 'LY:16; VLY:5; MO:2'}
{'model': 'convnext_tiny', 'true_class': 'PMY', 'support': 80, 'correct': 21, 'recall': 0.2625, 'top_wrong': 'MY:37; BL:20; MO:2'}
{'model': 'resnet50', 'true_class': 'BNE', 'support': 252, 'correct': 47, 'recall': 0.1865079365079365, 'top_wrong': 'SNE:171; MO:17; MMY:14'}
{'model': 'resnet50', 'true_class': 'MMY', 'support': 340, 'correct': 170, 'recall': 0.5, 'top_wrong': 'SNE:74; BNE:31; MY:27'}
{'model': 'resnet50', 'true_c